# Thesis analysis — WebArena-Verified Shopping

**Input:** `run-inventory.csv` and `analysis-results.csv`, exported by `npm run results:export`.

The notebook **does not run agents or WebArena**. It performs reproducible descriptive and paired analysis from **official evaluator scores**, without inventing missing results. Upload only sanitized CSV—not HAR, credentials, or private session logs.

Stages: pilot (6 × 4 × 2 = 48 planned episodes) or main (32 × 4 × 2 = 256 planned episodes). Pilot is a feasibility study; method significance tests are reserved for a complete main study.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binomtest

OUTPUT = Path("thesis_outputs")
OUTPUT.mkdir(exist_ok=True)
if not Path("run-inventory.csv").exists() or not Path("analysis-results.csv").exists():
    try:
        from google.colab import files
        print("Upload BOTH run-inventory.csv and analysis-results.csv from the same results:export.")
        files.upload()
    except ImportError:
        pass
assert Path("run-inventory.csv").exists(), "Missing run-inventory.csv"
assert Path("analysis-results.csv").exists(), "Missing analysis-results.csv"
inventory = pd.read_csv("run-inventory.csv")
analysis = pd.read_csv("analysis-results.csv")
required = {"stage","task_id","model","method","technical_valid","official_success","duration_ms","actions","probes","run_id"}
assert required.issubset(inventory.columns)
assert required.issubset(analysis.columns)
assert len(inventory) > 0
assert inventory["stage"].nunique() == 1, "Use one stage per notebook run"
stage = inventory["stage"].iloc[0]
assert stage in {"pilot","main"}
expected = 48 if stage == "pilot" else 256
assert len(inventory) == expected, f"Expected {expected} registered episodes, got {len(inventory)}"
keys = ["task_id","model","method"]
assert not inventory.duplicated(keys).any(), "Duplicate task/model/method combination"
valid = inventory["technical_valid"].astype(str).str.lower().eq("true")
assert len(analysis) == valid.sum(), "Validated analysis export differs from inventory"
completed = bool(valid.all())
print(f"Stage: {stage} | valid episodes: {valid.sum()}/{expected} | complete: {completed}")
if not completed:
    print("WARNING: INCOMPLETE. Descriptive results for valid episodes only; no confirmatory claims.")
df = analysis.copy()
df["official_success"] = df["official_success"].astype(str).str.lower().eq("true")
for col in ["duration_ms","actions","probes","model_calls"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
assert not df.duplicated(keys).any()


## 1. Coverage and run validity

The denominator for the planned study is always **48 or 256**. Unexecuted/invalid episodes are **not** coded as task failures.

In [ ]:
validity = inventory.groupby(["model","method"], as_index=False)["technical_valid"].apply(
    lambda col: int(col.astype(str).str.lower().eq("true").sum())
).rename(columns={"technical_valid":"valid_runs"})
validity.to_csv(OUTPUT/"run_validity.csv", index=False)
display(validity)
summary = pd.DataFrame([{
    "stage":stage, "expected":expected, "valid":int(valid.sum()),
    "invalid_or_missing":int((~valid).sum()), "complete":completed,
}])
summary.to_csv(OUTPUT/"coverage_summary.csv", index=False)
display(summary)


## 2. Official task success (descriptive)

These figures describe **valid evaluated runs only** when incomplete. They must not be called the overall success rate of the registered study unless coverage is complete.

In [ ]:
rates = df.groupby(["model","method"], as_index=False).agg(
    valid_n=("official_success","size"),
    successes=("official_success","sum"),
    success_rate=("official_success","mean"),
    median_duration_ms=("duration_ms","median"),
    median_actions=("actions","median"),
    median_probes=("probes","median"),
)
rates.to_csv(OUTPUT/"model_method_descriptives.csv", index=False)
display(rates)
pivot = rates.pivot(index="model",columns="method",values="success_rate").reindex(columns=["Baseline","Proposed"])
ax = pivot.mul(100).plot(kind="bar", rot=0, figsize=(9,4))
ax.set_ylabel("Official task success among valid runs (%)")
ax.set_xlabel("Model")
ax.set_title("WebArena-Verified Shopping: Baseline vs Proposed")
ax.legend(title="Method")
plt.tight_layout()
plt.savefig(OUTPUT/"success_by_model.png", dpi=300)
plt.savefig(OUTPUT/"success_by_model.pdf")
plt.show()


## 3. Runtime and action/probe efficiency

Median metrics avoid claiming a single typical runtime when runs are highly variable. Missing values remain missing.

In [ ]:
for metric,label,filename in [
    ("duration_ms","Median runtime (ms)","runtime_by_model.png"),
    ("actions","Median browser actions","actions_by_model.png"),
    ("probes","Median recorded evidence probes","probes_by_model.png"),
]:
    table = rates.pivot(index="model", columns="method", values="median_" + ("duration_ms" if metric=="duration_ms" else metric))
    table = table.reindex(columns=["Baseline","Proposed"])
    ax = table.plot(kind="bar", figsize=(9,4), rot=0)
    ax.set_ylabel(label)
    ax.set_title(label + ": Baseline vs Proposed")
    plt.tight_layout()
    plt.savefig(OUTPUT/filename, dpi=300)
    plt.show()


## 4. Paired exact McNemar and Holm correction — MAIN only

For each model, pair **Baseline and Proposed on the same task ID**. Count discordant pairs and use the exact binomial equivalent of McNemar (two-sided). Holm adjustment controls multiplicity across models. This notebook refuses confirmatory tests on a partial or pilot dataset.

In [ ]:
if stage != "main" or not completed:
    print("Confirmatory McNemar analysis unavailable: requires complete 256-run MAIN dataset.")
else:
    wide = df.pivot(index=["task_id","model"], columns="method", values="official_success")
    assert wide.shape == (128,2), "Missing or duplicate paired observations"
    assert wide[["Baseline","Proposed"]].notna().all().all()
    records = []
    for model, group in wide.groupby(level="model"):
        base = group["Baseline"].astype(bool)
        prop = group["Proposed"].astype(bool)
        b = int((base & ~prop).sum()) # Baseline-only
        c = int((~base & prop).sum()) # Proposed-only
        n = b+c
        p = binomtest(min(b,c),n,0.5,alternative="two-sided").pvalue if n else 1.0
        records.append({
            "model":model, "baseline_only":b, "proposed_only":c,
            "both_success":int((base & prop).sum()),
            "both_failure":int((~base & ~prop).sum()),
            "paired_rate_difference":float(prop.mean()-base.mean()),
            "discordant_n":n, "mcnemar_exact_p":p,
        })
    paired = pd.DataFrame(records).sort_values("mcnemar_exact_p")
    m = len(paired)
    adjusted = np.maximum.accumulate(np.minimum(1,np.array(paired["mcnemar_exact_p"])*np.arange(m,0,-1)))
    paired["holm_adjusted_p"] = adjusted
    paired.to_csv(OUTPUT/"main_paired_mcnemar.csv", index=False)
    display(paired)
    print("Interpret alongside effect size, task coverage, floor effects and uncertainty.")


## 5. Download outputs for Bab IV

Figures use 300 DPI PNG and a PDF for the primary graph. Tables are exported as CSV. The notebook stores **no screenshots, account credentials, HAR, or hidden benchmark reference answers**.

In [ ]:
from zipfile import ZipFile
archive = Path("thesis_outputs.zip")
with ZipFile(archive,"w") as zf:
    for p in sorted(OUTPUT.iterdir()):
        if p.is_file():
            zf.write(p, arcname=p.name)
print("Created", archive, "with", len(list(OUTPUT.iterdir())), "files")
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass
